# Assignment: Hypothesis Testing with Z-Test and t-Test
### Left-tailed, right-tailed and two-tailed tests on a student dataset

**Total marks:** 100  **Submit:** one `.ipynb` file

**What you will do**

You are given a small dataset of **40 college students**. Answer **6 questions**:

| Question | Test | Tail |
|---|---|---|
| Q1 | Z-test | Left |
| Q2 | Z-test | Right |
| Q3 | Z-test | Two |
| Q4 | t-test | Left |
| Q5 | t-test | Right |
| Q6 | t-test | Two |

Then fill in the summary table in Q7.

**Instructions**
1. Run the two starter cells first. **Do not change them.**
2. Write your code only in the cells marked `YOUR CODE HERE`.
3. Write your answers in the cells marked **Your answer**.
4. Use **α = 0.05** for every question.
5. Allowed libraries: `numpy`, `pandas`, `scipy.stats`, `matplotlib`.
6. Compute every number from the dataset. Do not type in the answers by hand.

## Quick Reference

| | Z-test | t-test |
|---|---|---|
| Use when | Population σ is **known** | Population σ is **unknown** |
| Formula | $z = \dfrac{\bar{x} - \mu_0}{\sigma / \sqrt{n}}$ | $t = \dfrac{\bar{x} - \mu_0}{s / \sqrt{n}}$, $df = n - 1$ |

| Tail | $H_1$ | p-value (Z) | p-value (t) | Critical value (Z, α = 0.05) |
|---|---|---|---|---|
| Left | $\mu < \mu_0$ | `stats.norm.cdf(z)` | `stats.t.cdf(t, dof)` | −1.645 |
| Right | $\mu > \mu_0$ | `1 - stats.norm.cdf(z)` | `1 - stats.t.cdf(t, dof)` | +1.645 |
| Two | $\mu \neq \mu_0$ | `2 * (1 - stats.norm.cdf(abs(z)))` | `2 * (1 - stats.t.cdf(abs(t), dof))` | ±1.960 |

**Decision rule:** if **p-value ≤ α**, reject $H_0$. Otherwise, fail to reject $H_0$.

For the sample standard deviation use `df['column'].std()`. pandas uses `ddof=1` by default, which is what the t-test needs.

⚠️ The dataset is stored in a variable called `df`. Store the degrees of freedom in a variable called **`dof`**, not `df`, or you will overwrite the dataset.

## Setup and Dataset

In [19]:
import pandas as pd
print(pd.__version__)

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# ---------------- STARTER CODE – DO NOT MODIFY ----------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

alpha = 0.05
print("Libraries loaded. alpha =", alpha)

In [ ]:
# ---------------- STARTER CODE – DO NOT MODIFY ----------------
# Dataset: 40 students from three sections of a college course
df = pd.DataFrame({
    'student_id':  [f'S{i:02d}' for i in range(1, 41)],
    'section':     ['A'] * 14 + ['B'] * 13 + ['C'] * 13,
    'sleep_hours': [6.9, 5.5, 7.4, 7.6, 4.5, 5.2, 6.7, 6.3, 6.6, 5.7, 7.6, 7.5, 6.7, 7.8,
                    7.1, 5.7, 7.0, 5.5, 7.6, 6.5, 6.4, 5.9, 7.9, 6.4, 6.1, 6.2, 7.2,
                    7.0, 7.1, 7.1, 9.0, 6.2, 6.0, 5.7, 7.3, 7.8, 6.5, 5.7, 5.7, 7.3],
    'screen_time': [6.5, 6.2, 4.5, 5.8, 5.6, 5.8, 6.7, 5.8, 6.4, 5.5, 5.9, 6.3, 3.4, 5.0,
                    4.8, 4.6, 5.1, 7.5, 4.2, 6.8, 3.1, 5.0, 5.7, 6.3, 6.4, 6.6, 5.0,
                    4.8, 6.7, 5.2, 3.7, 3.9, 4.2, 6.1, 5.6, 6.4, 4.9, 5.7, 6.3, 5.0],
    'exam_score':  [71, 60, 63, 63, 55, 71, 62, 67, 71, 71, 73, 66, 62, 66,
                    50, 52, 53, 57, 70, 57, 63, 79, 63, 74, 57, 64, 57,
                    63, 75, 49, 71, 69, 61, 52, 67, 61, 69, 67, 83, 64],
    'attendance':  [75, 82, 82, 89, 86, 83, 90, 74, 77, 75, 79, 73, 85, 80,
                    72, 75, 83, 86, 93, 98, 83, 75, 68, 83, 76, 79, 77,
                    80, 87, 82, 80, 75, 71, 78, 81, 92, 82, 87, 78, 74],
    'study_hours': [2.2, 2.4, 4.7, 2.3, 3.7, 2.3, 3.7, 3.3, 2.9, 3.0, 2.5, 3.4, 2.6, 2.0,
                    2.0, 3.1, 4.3, 3.1, 2.9, 3.2, 4.0, 3.2, 2.7, 3.9, 3.3, 4.2, 3.1,
                    2.0, 1.9, 4.3, 4.4, 2.9, 2.7, 4.2, 2.1, 2.3, 3.5, 2.7, 3.0, 2.9],
    'commute_min': [34, 42, 33, 37, 18, 32, 26, 23, 26, 30, 38, 23, 32, 29,
                    30, 39, 36, 41, 31, 27, 30, 34, 33, 24, 33, 34, 50,
                    45, 26, 30, 22, 28, 34, 40, 27, 27, 17, 31, 25, 28],
})

print("Shape:", df.shape)
print(df['section'].value_counts().sort_index().to_string())
df.head(10)

**Data dictionary**

| Column | Meaning |
|---|---|
| `student_id` | Student ID |
| `section` | Class section: A (14 students), B (13), C (13) |
| `sleep_hours` | Average sleep per night (hours) |
| `screen_time` | Average daily screen time (hours) |
| `exam_score` | Final exam score (out of 100) |
| `attendance` | Attendance (%) |
| `study_hours` | Average self-study per day (hours) |
| `commute_min` | One-way commute time (minutes) |

Explore the data before you start (optional):

In [ ]:
# ===================== OPTIONAL – EXPLORE THE DATA =====================
# OPTIONAL: explore the dataset, e.g. df.describe() or df.groupby('section').mean(numeric_only=True)

---
# Part A – Z-Tests (σ known, use all 40 students)

## Q1. Z-Test – Left-Tailed

A health report says college students sleep **7 hours** per night on average, with a known population standard deviation of **σ = 1.2 hours**.
Using `sleep_hours` for **all 40 students**, test at α = 0.05 whether students sleep **less than 7 hours** on average.

**Steps:** (a) write $H_0$, $H_1$ and the tail, (b) find $n$ and $\bar{x}$, (c) compute $z$, (d) find the critical value and the p-value, (e) make a decision and write a conclusion.

*(15 marks)*

In [ ]:
# Q1: Z-Test - Left-Tailed

data = df['sleep_hours']

mu_0 = 7
sigma = 1.2
alpha = 0.05

n = len(data)
mean = data.mean()

z = (mean - mu_0) / (sigma / np.sqrt(n))

critical_value = stats.norm.ppf(alpha)
p_value = stats.norm.cdf(z)

print("H0: μ = 7")
print("H1: μ < 7")
print("Tail: Left-tailed")
print("Sample size:", n)
print("Sample mean:", round(mean, 4))
print("Z statistic:", round(z, 4))
print("Critical value:", round(critical_value, 4))
print("p-value:", round(p_value, 4))

if p_value <= alpha:
    print("Decision: Reject H0")
    print("Conclusion: Students sleep less than 7 hours on average.")
else:
    print("Decision: Fail to reject H0")
    print("Conclusion: There is not enough evidence that students sleep less than 7 hours.")

**Your answer – Q1**

- **$H_0$:** $\mu = 7$ hours
- **$H_1$:** $\mu < 7$ hours
- **Tail (left / right / two):** Left-tailed
- **Test statistic:** $z = -1.8578$
- **Critical value:** $-1.6449$
- **p-value:** $0.0316$
- **Decision (Reject $H_0$ / Fail to reject $H_0$):** Reject $H_0$
- **Conclusion in words:** At the 5% level, there is sufficient evidence that the students sleep less than 7 hours per night on average.

## Q2. Z-Test – Right-Tailed

A national survey says young adults spend **5 hours** a day on screens, with a known population standard deviation of **σ = 1.5 hours**.
Using `screen_time` for **all 40 students**, test at α = 0.05 whether these students' screen time is **more than 5 hours** a day.

**Steps:** (a) write $H_0$, $H_1$ and the tail, (b) find $n$ and $\bar{x}$, (c) compute $z$, (d) find the critical value and the p-value, (e) make a decision and write a conclusion.

*(15 marks)*

In [ ]:
# Q2: Z-Test - Right-Tailed

data = df['screen_time']

mu_0 = 5
sigma = 1.5
alpha = 0.05

n = len(data)
mean = data.mean()

z = (mean - mu_0) / (sigma / np.sqrt(n))

critical_value = stats.norm.ppf(1 - alpha)
p_value = 1 - stats.norm.cdf(z)

print("H0: μ = 5")
print("H1: μ > 5")
print("Tail: Right-tailed")
print("Sample size:", n)
print("Sample mean:", round(mean, 4))
print("Z statistic:", round(z, 4))
print("Critical value:", round(critical_value, 4))
print("p-value:", round(p_value, 4))

if p_value <= alpha:
    print("Decision: Reject H0")
    print("Conclusion: Students spend more than 5 hours on screens on average.")
else:
    print("Decision: Fail to reject H0")
    print("Conclusion: There is not enough evidence that students spend more than 5 hours on screens.")

**Your answer – Q2**

- **$H_0$:** $\mu = 5$ hours
- **$H_1$:** $\mu > 5$ hours
- **Tail (left / right / two):** Right-tailed
- **Test statistic:** $z = 2.0028$
- **Critical value:** $1.6449$
- **p-value:** $0.0226$
- **Decision (Reject $H_0$ / Fail to reject $H_0$):** Reject $H_0$
- **Conclusion in words:** At the 5% level, there is sufficient evidence that the students spend more than 5 hours per day on screens on average.

## Q3. Z-Test – Two-Tailed

Over many years, the course's final exam has had a mean of **65** with a known standard deviation of **σ = 10**.
Using `exam_score` for **all 40 students**, test at α = 0.05 whether this year's mean score is **different from 65**.

**Steps:** (a) write $H_0$, $H_1$ and the tail, (b) find $n$ and $\bar{x}$, (c) compute $z$, (d) find the critical value and the p-value, (e) make a decision and write a conclusion.

*(15 marks)*

In [ ]:
# Q3: Z-Test - Two-Tailed

data = df['exam_score']

mu_0 = 65
sigma = 10
alpha = 0.05

n = len(data)
mean = data.mean()

z = (mean - mu_0) / (sigma / np.sqrt(n))

critical_value = stats.norm.ppf(1 - alpha / 2)
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

print("H0: μ = 65")
print("H1: μ ≠ 65")
print("Tail: Two-tailed")
print("Sample size:", n)
print("Sample mean:", round(mean, 4))
print("Z statistic:", round(z, 4))
print("Critical values:", round(-critical_value, 4), "and", round(critical_value, 4))
print("p-value:", round(p_value, 4))

if p_value <= alpha:
    print("Decision: Reject H0")
    print("Conclusion: The mean exam score is different from 65.")
else:
    print("Decision: Fail to reject H0")
    print("Conclusion: There is not enough evidence that the mean exam score differs from 65.")

**Your answer – Q3**

- **$H_0$:** $\mu = 65$
- **$H_1$:** $\mu \ne 65$
- **Tail (left / right / two):** Two-tailed
- **Test statistic:** $z = -0.5060$
- **Critical value:** $\pm 1.9600$
- **p-value:** $0.6129$
- **Decision (Reject $H_0$ / Fail to reject $H_0$):** Fail to reject $H_0$
- **Conclusion in words:** At the 5% level, there is insufficient evidence that this year's mean exam score differs from 65.

---
# Part B – t-Tests (σ unknown, use one section only)

Select one section like this: `df[df['section'] == 'B']['attendance']`

## Q4. t-Test – Left-Tailed

The class teacher of **Section B** believes her students' attendance is **below 83 %**. The population standard deviation is **not known**.
Using `attendance` for **Section B only**, test at α = 0.05 whether the mean attendance is **less than 83 %**.

**Steps:** (a) write $H_0$, $H_1$ and the tail, (b) select the rows for the section, then find $n$, $\bar{x}$, $s$ and $df$, (c) compute $t$, (d) find the critical value and the p-value, (e) make a decision and write a conclusion, (f) check your answer with `stats.ttest_1samp(data, popmean, alternative=...)`.

*(15 marks)*

In [ ]:
# Q4: t-Test - Left-Tailed

data = df[df['section'] == 'B']['attendance']

mu_0 = 83
alpha = 0.05

n = len(data)
mean = data.mean()
s = data.std()
dof = n - 1

t = (mean - mu_0) / (s / np.sqrt(n))

critical_value = stats.t.ppf(alpha, dof)
p_value = stats.t.cdf(t, dof)

print("H0: μ = 83")
print("H1: μ < 83")
print("Tail: Left-tailed")
print("Sample size:", n)
print("Sample mean:", round(mean, 4))
print("Sample standard deviation:", round(s, 4))
print("Degrees of freedom:", dof)
print("t statistic:", round(t, 4))
print("Critical value:", round(critical_value, 4))
print("p-value:", round(p_value, 4))

if p_value <= alpha:
    print("Decision: Reject H0")
    print("Conclusion: Section B attendance is less than 83% on average.")
else:
    print("Decision: Fail to reject H0")
    print("Conclusion: There is not enough evidence that Section B attendance is less than 83%.")

result = stats.ttest_1samp(data, mu_0, alternative='less')
print("SciPy check:", result)

**Your answer – Q4**

- **$H_0$:** $\mu = 83\%$
- **$H_1$:** $\mu < 83\%$
- **Tail (left / right / two):** Left-tailed
- **Test statistic:** $t = -1.0307$ ($df = 12$)
- **Critical value:** $-1.7823$
- **p-value:** $0.1615$
- **Decision (Reject $H_0$ / Fail to reject $H_0$):** Fail to reject $H_0$
- **Conclusion in words:** At the 5% level, there is insufficient evidence that Section B's mean attendance is below 83%.

## Q5. t-Test – Right-Tailed

The college recommends at least **2.5 hours** of self-study per day. The teacher of **Section A** claims her students study **more than 2.5 hours** a day on average. σ is **not known**.
Using `study_hours` for **Section A only**, test the claim at α = 0.05.

**Steps:** (a) write $H_0$, $H_1$ and the tail, (b) select the rows for the section, then find $n$, $\bar{x}$, $s$ and $df$, (c) compute $t$, (d) find the critical value and the p-value, (e) make a decision and write a conclusion, (f) check your answer with `stats.ttest_1samp(data, popmean, alternative=...)`.

*(15 marks)*

In [ ]:
# Q5: t-Test - Right-Tailed

data = df[df['section'] == 'A']['study_hours']

mu_0 = 2.5
alpha = 0.05

n = len(data)
mean = data.mean()
s = data.std()
dof = n - 1

t = (mean - mu_0) / (s / np.sqrt(n))

critical_value = stats.t.ppf(1 - alpha, dof)
p_value = 1 - stats.t.cdf(t, dof)

print("H0: μ = 2.5")
print("H1: μ > 2.5")
print("Tail: Right-tailed")
print("Sample size:", n)
print("Sample mean:", round(mean, 4))
print("Sample standard deviation:", round(s, 4))
print("Degrees of freedom:", dof)
print("t statistic:", round(t, 4))
print("Critical value:", round(critical_value, 4))
print("p-value:", round(p_value, 4))

if p_value <= alpha:
    print("Decision: Reject H0")
    print("Conclusion: Section A students study more than 2.5 hours on average.")
else:
    print("Decision: Fail to reject H0")
    print("Conclusion: There is not enough evidence that Section A students study more than 2.5 hours.")

result = stats.ttest_1samp(data, mu_0, alternative='greater')
print("SciPy check:", result)

**Your answer – Q5**

- **$H_0$:** $\mu = 2.5$ hours
- **$H_1$:** $\mu > 2.5$ hours
- **Tail (left / right / two):** Right-tailed
- **Test statistic:** $t = 2.1185$ ($df = 13$)
- **Critical value:** $1.7709$
- **p-value:** $0.0270$
- **Decision (Reject $H_0$ / Fail to reject $H_0$):** Reject $H_0$
- **Conclusion in words:** At the 5% level, there is sufficient evidence that Section A students study more than 2.5 hours per day on average.

## Q6. t-Test – Two-Tailed

The transport office plans the college bus timetable assuming an average one-way commute of **35 minutes**. σ is **not known**.
Using `commute_min` for **Section C only**, test at α = 0.05 whether the mean commute time is **different from 35 minutes**.

**Steps:** (a) write $H_0$, $H_1$ and the tail, (b) select the rows for the section, then find $n$, $\bar{x}$, $s$ and $df$, (c) compute $t$, (d) find the critical value and the p-value, (e) make a decision and write a conclusion, (f) check your answer with `stats.ttest_1samp(data, popmean, alternative=...)`.

*(15 marks)*

In [ ]:
# Q6: t-Test - Two-Tailed

data = df[df['section'] == 'C']['commute_min']

mu_0 = 35
alpha = 0.05

n = len(data)
mean = data.mean()
s = data.std()
dof = n - 1

t = (mean - mu_0) / (s / np.sqrt(n))

critical_value = stats.t.ppf(1 - alpha / 2, dof)
p_value = 2 * (1 - stats.t.cdf(abs(t), dof))

print("H0: μ = 35")
print("H1: μ ≠ 35")
print("Tail: Two-tailed")
print("Sample size:", n)
print("Sample mean:", round(mean, 4))
print("Sample standard deviation:", round(s, 4))
print("Degrees of freedom:", dof)
print("t statistic:", round(t, 4))
print("Critical values:", round(-critical_value, 4), "and", round(critical_value, 4))
print("p-value:", round(p_value, 4))

if p_value <= alpha:
    print("Decision: Reject H0")
    print("Conclusion: The mean commute time is different from 35 minutes.")
else:
    print("Decision: Fail to reject H0")
    print("Conclusion: There is not enough evidence that the mean commute time differs from 35 minutes.")

result = stats.ttest_1samp(data, mu_0, alternative='two-sided')
print("SciPy check:", result)

**Your answer – Q6**

- **$H_0$:** $\mu = 35$ minutes
- **$H_1$:** $\mu \ne 35$ minutes
- **Tail (left / right / two):** Two-tailed
- **Test statistic:** $t = -2.8611$ ($df = 12$)
- **Critical value:** $\pm 2.1788$
- **p-value:** $0.0143$
- **Decision (Reject $H_0$ / Fail to reject $H_0$):** Reject $H_0$
- **Conclusion in words:** At the 5% level, there is sufficient evidence that Section C's mean commute time differs from 35 minutes.

---
## Q7. Summary Table *(10 marks)*

Fill in the table with your results, then answer the two short questions below it.

**Your answer – Q7**

| Q | Test | Tail | Statistic | Critical value | p-value | Decision |
|---|---|---|---|---|---|---|
| Q1 |Z-test |Left |-1.8578 |-1.6449 |0.0316 |Reject H₀ |
| Q2 |Z-test |Right |2.0028 |1.6449 |0.0226 |Reject H₀ |
| Q3 |Z-test |Two |-0.5060 |+-1.9600 |0.6129 |Fail to reject H₀ |
| Q4 | t-test | Left | -1.0307 | -1.7823 | 0.1615 | Fail to reject H₀ |
| Q5 | t-test | Right | 2.1185 | 1.7709 | 0.0270 | Reject H₀ |
| Q6 | t-test | Two | -2.8611 | ±2.1788 | 0.0143 | Reject H₀ |

**(a)** Q1–Q3 use Z-tests because the population standard deviations are given. Q4–Q6 use t-tests because the population standard deviations are unknown and are estimated from each sample.

**(b)** No. Failing to reject $H_0$ means the sample does not provide enough evidence against it at the chosen significance level; it does not prove that $H_0$ is true.

---
## Submission Guidelines

1. Complete every `YOUR CODE HERE` cell and every **Your answer** cell.
2. Before submitting, click **Kernel → Restart & Run All** (in Colab: **Runtime → Restart session and run all**). Make sure there are no errors and all outputs are visible.
3. Do not delete or change the starter cells.
4. Save the file as `RollNumber_Name_ZtTest.ipynb` (for example `23CS1042_Ananya_ZtTest.ipynb`).
5. Submit **only the `.ipynb` file** on the LMS before the deadline. PDF, HTML or `.py` files are not accepted.
6. This is an individual assignment. Do not copy code or answers from classmates.

**Marking:** Q1–Q6 are worth 15 marks each: hypotheses and tail (3), correct calculations (6), decision (3) and conclusion in words (3). Q7 is worth 10 marks. Total = 100.